In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install odfpy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 717.0/717.0 kB 9.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for odfpy: filename=odfpy-1.4.1-py2.py3-none-any.whl size=160672 sha256=967452fa6295c3933dc0782bade1608242d4605688a89a88e0b9623ba8a57a29
  Stored in directory: /root/.cache/pip/wheels/d6/1d/c8/8c29be1d73ca42d15977c75193d9f39a98499413c2838ac54c
Successfully built odfpy


In [3]:
import pandas as pd

# Đọc dữ liệu từ các file CSV
output_user_stats = pd.read_csv('/content/drive/MyDrive/CS114.P11/predict/QT/features.csv')
ck_public = pd.read_excel('/content/drive/MyDrive/CS114.P11/predict/TBTL/tbtl-public.ods', engine="odf")

# Kiểm tra thông tin về dữ liệu
print(output_user_stats.head())
print(ck_public.head())

                                   username  total_pre_score  total_problems  \
0  00b6dd4fc7eb817e03708c532016ef30ce564a61           809110              46   
1  00bef8afee8f3c595d535c9c03c490cac1a4f021          1421535              72   
2  01122b3ef7e59b84189e65985305f575d6bdf83c          1164882              58   
3  0134f9f410c65ad0e8c2254a7e9288670e02a183           595276              47   
4  013de369c439ab0ead8aa7da64423aa395a8be39           692766              44   

   total_days  total_submissions  total_late_days  avg_time   avg_memory  \
0          14                147                0  0.322517  2289.850340   
1          20                259                0  0.253514  2946.625483   
2          25                195                0  0.201436  2054.256410   
3          13                100                0  0.066000   374.480000   
4           8                107                3  0.743832  4987.813084   

   total_assignments  total_score_submissions  total_error_sub

In [4]:
# Lọc những người dùng chưa có điểm TBTL trong ck-public.csv
users_with_scores = ck_public['username'].tolist()
users_no_score = output_user_stats[~output_user_stats['username'].isin(users_with_scores)]

# Lọc những người dùng có dữ liệu điểm TBTL
users_with_data = output_user_stats[output_user_stats['username'].isin(users_with_scores)]

# Kiểm tra lại dữ liệu đã lọc
print(users_no_score.head())
print(users_with_data.head())

                                     username  total_pre_score  \
394  410357eb9129023509cfaf8d38be61c050bb3b05           206426   
625  67212308d026508fd5b6942ffbbdd7b0be2e89de                0   
801  84a17972cc6d29489bbe205a9e7feb8745726fbc          1235612   
802  84b6b2d70924066c8345f2bc2281791ae3188da2          1001865   
803  851d9a4b9b8e236f2d62282ddf06fae57b7d9492          1737548   

     total_problems  total_days  total_submissions  total_late_days  avg_time  \
394              16           5                 43                0  0.000000   
625               0           1                  8                0  0.000000   
801              98          32                182                0  0.003571   
802              77          30                160                0  0.349937   
803              90          31                389                0  0.128792   

      avg_memory  total_assignments  total_score_submissions  \
394     0.000000                  2                 

In [7]:
import pandas as pd
from sklearn.ensemble import VotingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

# Chọn các cột đặc trưng và cột mục tiêu
X = users_with_data[['total_pre_score', 'total_problems', 'total_days', 'total_submissions', 'avg_time', 'avg_memory', 'total_assignments', 'total_score_submissions', 'total_error_submissions', 'final_total_score']]
y = ck_public.set_index('username').loc[users_with_data['username'], 'TBTL']

# Chia dữ liệu thành tập huấn luyện và tập kiểm tra
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=40)

# Khởi tạo các mô hình thành phần
linear_reg = LinearRegression()
random_forest = RandomForestRegressor(random_state=42)
gradient_boost = GradientBoostingRegressor(random_state=42)

# Tạo Voting Regressor
voting_regressor = VotingRegressor(estimators=[
    ('lr', linear_reg),
    ('rf', random_forest),
    ('gb', gradient_boost)
])

# Huấn luyện mô hình Voting Regressor
voting_regressor.fit(X_train, y_train)

# Dự đoán trên tập kiểm tra
y_pred = voting_regressor.predict(X_test)

# Đánh giá mô hình
mse = mean_squared_error(y_test, y_pred)
print(f'Mean Squared Error: {mse}')

# Dự đoán cho những người dùng chưa có điểm
X_no_score = users_no_score[['total_pre_score', 'total_problems', 'total_days', 'total_submissions', 'avg_time', 'avg_memory', 'total_assignments', 'total_score_submissions', 'total_error_submissions', 'final_total_score']]
predicted_scores = voting_regressor.predict(X_no_score)

# Thêm kết quả dự đoán vào DataFrame
users_no_score['predicted_TBTL'] = predicted_scores

# Kiểm tra kết quả
print(users_no_score[['username', 'predicted_TBTL']].head())

# Xuất kết quả dự đoán vào file CSV
users_no_score[['username', 'predicted_TBTL']].to_csv('/content/drive/MyDrive/CS114.P11/predict/TBTL/predicted_scores111.csv', index=False)

# Định nghĩa hàm custom_round
def custom_round(value):
    decimal_part = value - int(value)  # Lấy phần thập phân
    if decimal_part < 0.25:
        return int(value)
    elif decimal_part < 0.75:
        return int(value) + 0.5
    else:
        return int(value) + 1

# Đọc lại dữ liệu dự đoán
predicted_tbtl_path = '/content/drive/MyDrive/CS114.P11/predict/TBTL/predicted_scores111.csv'
predicted_tbtl = pd.read_csv(predicted_tbtl_path)

# Áp dụng làm tròn
predicted_tbtl['predicted_TBTL'] = predicted_tbtl['predicted_TBTL'].apply(custom_round)

# Lưu lại file đã làm tròn
rounded_output_path = '/content/drive/MyDrive/CS114.P11/predict/TBTL/predicted_ck111_rounded.csv'
predicted_tbtl.to_csv(rounded_output_path, index=False)

print(f"Rounded TBTL scores saved to {rounded_output_path}")


Mean Squared Error: 0.6657961010032502
                                     username  predicted_TBTL
394  410357eb9129023509cfaf8d38be61c050bb3b05        7.305107
625  67212308d026508fd5b6942ffbbdd7b0be2e89de        7.526846
801  84a17972cc6d29489bbe205a9e7feb8745726fbc        7.687694
802  84b6b2d70924066c8345f2bc2281791ae3188da2        7.539706
803  851d9a4b9b8e236f2d62282ddf06fae57b7d9492        7.846982
Rounded TBTL scores saved to /content/drive/MyDrive/CS114.P11/predict/TBTL/predicted_ck111_rounded.csv


<ipython-input-7-ce9c88fb30b7>:43: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  users_no_score['predicted_TBTL'] = predicted_scores


In [8]:
# Khởi tạo và huấn luyện mô hình hồi quy tuyến tính
model = LinearRegression()
model.fit(X_train, y_train)

# Dự đoán trên tập kiểm tra
y_pred = model.predict(X_test)

# Đánh giá mô hình
mse = mean_squared_error(y_test, y_pred)
print(f'Mean Squared Error: {mse}')

Mean Squared Error: 0.6724187303665821
